<a href="https://colab.research.google.com/github/NavameyPJ/Credit-Risk-Scorecard-PD-LGD-EAD/blob/main/Credit_Risk_Scorecard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# 1. Direct web link to the dataset
url = 'https://raw.githubusercontent.com/mesfind/datasets/master/cs-training.csv'

# 2. Read dataset directly from the web into memory
df = pd.read_csv(url, index_col=0)

# 3. Clean column names (strip trailing spaces)
df.columns = df.columns.str.strip()

# 4. Build text summary
summary_text = f"""=== CREDIT RISK DATASET STRUCTURE SUMMARY ===
Total Records (Borrowers): {df.shape[0]:,}
Total Features (Columns): {df.shape[1]}

--- COLUMN NAMES & DATA TYPES ---
{df.dtypes.to_string()}

--- MISSING VALUES COUNT ---
{df.isnull().sum().to_string()}

--- FIRST 5 ROWS PREVIEW ---
{df.head().to_string()}
"""

# 5. Output summary on screen
print(summary_text)

# 6. Save as a .txt file inside your Google Colab workspace
with open('dataset_info.txt', 'w') as f:
    f.write(summary_text)

print("\n Saved dataset structure into 'dataset_info.txt'.")

=== CREDIT RISK DATASET STRUCTURE SUMMARY ===
Total Records (Borrowers): 150,000
Total Features (Columns): 0

--- COLUMN NAMES & DATA TYPES ---
Series([], )

--- MISSING VALUES COUNT ---
Series([], )

--- FIRST 5 ROWS PREVIEW ---
Empty DataFrame
Columns: []
Index: [1;1;0.766126609;45;2;0.802982129;9120;13;0;6;0;2, 2;0;0.957151019;40;0;0.121876201;2600;4;0;0;0;1, 3;0;0.65818014;38;1;0.085113375;3042;2;1;0;0;0, 4;0;0.233809776;30;0;0.036049682;3300;5;0;0;0;0, 5;0;0.9072394;49;1;0.024925695;63588;7;0;1;0;0]


 Saved dataset structure into 'dataset_info.txt'.


In [3]:
import pandas as pd

# 1. Dataset URL
url = 'https://raw.githubusercontent.com/mesfind/datasets/master/cs-training.csv'

# 2. Add sep=';' so pandas correctly splits columns on semicolons
df = pd.read_csv(url, index_col=0, sep=';')

# 3. Clean hidden trailing whitespace from column titles
df.columns = df.columns.str.strip()

# 4. Generate structured metadata summary
summary_text = f"""=== CREDIT RISK DATASET STRUCTURE SUMMARY ===
Total Records (Borrowers): {df.shape[0]:,}
Total Features (Columns): {df.shape[1]}

--- COLUMN NAMES & DATA TYPES ---
{df.dtypes.to_string()}

--- MISSING VALUES COUNT ---
{df.isnull().sum().to_string()}

--- FIRST 5 ROWS PREVIEW ---
{df.head().to_string()}
"""

# 5. Output summary on screen
print(summary_text)

# 6. Overwrite dataset_info.txt with the correct 11-column layout
with open('dataset_info.txt', 'w') as f:
    f.write(summary_text)

print("\n Successfully updated dataset_info.txt!")

=== CREDIT RISK DATASET STRUCTURE SUMMARY ===
Total Records (Borrowers): 150,000
Total Features (Columns): 11

--- COLUMN NAMES & DATA TYPES ---
SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64

--- MISSING VALUES COUNT ---
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLin

In [4]:
import numpy as np

# 1. Fill missing MonthlyIncome with the median income of all borrowers
median_income = df['MonthlyIncome'].median()
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(median_income)

# 2. Fill missing NumberOfDependents with 0 (the mode)
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# 3. Cap extreme credit utilization values above 1.0 (100% of credit limit) down to 1.0
df['RevolvingUtilizationOfUnsecuredLines'] = np.where(
    df['RevolvingUtilizationOfUnsecuredLines'] > 1, 1, df['RevolvingUtilizationOfUnsecuredLines']
)

# 4. Verify that missing values are completely resolved across all columns
print("=== MISSING VALUES AFTER CLEANING ===")
print(df.isnull().sum())
print("\n Cleaned dataset ready with 0 missing values!")

=== MISSING VALUES AFTER CLEANING ===
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

 Cleaned dataset ready with 0 missing values!


In [5]:
import numpy as np
import pandas as pd

target_col = 'SeriousDlqin2yrs'

def calculate_iv(dataset, feature, target):
    # Copy feature and target columns
    df_temp = dataset[[feature, target]].copy()

    # Rank continuous values into 10 percentile bins (deciles)
    try:
        df_temp['bin'] = pd.qcut(df_temp[feature].rank(method='first'), q=10, duplicates='drop')
    except Exception:
        return 0.0

    # Group by bin and calculate non-default vs default counts
    grouped = df_temp.groupby('bin', observed=False)[target].agg(
        Total='count',
        Defaults='sum'
    )
    grouped['NonDefaults'] = grouped['Total'] - grouped['Defaults']

    total_non_defaults = grouped['NonDefaults'].sum()
    total_defaults = grouped['Defaults'].sum()

    if total_non_defaults == 0 or total_defaults == 0:
        return 0.0

    dist_non_defaults = grouped['NonDefaults'] / total_non_defaults
    dist_defaults = grouped['Defaults'] / total_defaults

    # Calculate WOE and IV
    woe = np.log((dist_non_defaults + 1e-5) / (dist_defaults + 1e-5))
    iv = (dist_non_defaults - dist_defaults) * woe

    return float(iv.sum())

# Calculate IV for all 10 financial features
features = [col for col in df.columns if col != target_col]
iv_results = []

for feature in features:
    iv_val = calculate_iv(df, feature, target_col)
    iv_results.append({'Feature': feature, 'Information Value (IV)': round(iv_val, 4)})

# Display ranked results table
iv_df = pd.DataFrame(iv_results).sort_values(by='Information Value (IV)', ascending=False).reset_index(drop=True)
print("=== FEATURE PREDICTIVE POWER (INFORMATION VALUE) ===")
print(iv_df)

=== FEATURE PREDICTIVE POWER (INFORMATION VALUE) ===
                                Feature  Information Value (IV)
0  RevolvingUtilizationOfUnsecuredLines                  1.1132
1  NumberOfTime30-59DaysPastDueNotWorse                  0.6426
2               NumberOfTimes90DaysLate                  0.5864
3  NumberOfTime60-89DaysPastDueNotWorse                  0.3739
4                                   age                  0.2561
5       NumberOfOpenCreditLinesAndLoans                  0.0786
6                             DebtRatio                  0.0744
7                         MonthlyIncome                  0.0735
8          NumberRealEstateLoansOrLines                  0.0488
9                    NumberOfDependents                  0.0350


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Define Features (X) and Target Variable (y)
features = [col for col in df.columns if col != 'SeriousDlqin2yrs']
X = df[features]
y = df['SeriousDlqin2yrs']

# 2. Split data into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Initialize and train the Logistic Regression model
pd_model = LogisticRegression(max_iter=1000)
pd_model.fit(X_train, y_train)

# 4. Predict Probability of Default (PD) on the test set
y_pred_proba = pd_model.predict_proba(X_test)[:, 1]

# 5. Evaluate prediction quality using ROC-AUC Score
auc_score = roc_auc_score(y_test, y_pred_proba)

print("=== STEP 4 MODEL TRAINING RESULTS ===")
print(f"Training Data Size: {X_train.shape[0]:,} borrowers")
print(f"Testing Data Size:  {X_test.shape[0]:,} borrowers")
print(f"\nModel ROC-AUC Score: {auc_score:.4f}")

=== STEP 4 MODEL TRAINING RESULTS ===
Training Data Size: 120,000 borrowers
Testing Data Size:  30,000 borrowers

Model ROC-AUC Score: 0.8039


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Build pipeline: Standardize feature scales first, then train Logistic Regression
scaled_pd_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

# 2. Fit pipeline on training data
scaled_pd_model.fit(X_train, y_train)

# 3. Generate clean Probability of Default (PD) predictions for the 30,000 test set borrowers
clean_pd_predictions = scaled_pd_model.predict_proba(X_test)[:, 1]

# Verify ROC-AUC score after scaling
clean_auc = roc_auc_score(y_test, clean_pd_predictions)

# 4. Construct Bank Portfolio Evaluation Table
portfolio_df = pd.DataFrame({
    'Actual_Default': y_test.values,
    'PD': clean_pd_predictions
})

# 5. Define Basel III Parameters
LGD = 0.45   # 45% Loss Given Default
EAD = 10000  # $10,000 Exposure at Default per credit line

# 6. Calculate Expected Loss per borrower
portfolio_df['Expected_Loss'] = portfolio_df['PD'] * LGD * EAD

# 7. Portfolio Level Loss Metrics
total_test_borrowers = len(portfolio_df)
total_credit_exposure = total_test_borrowers * EAD
total_expected_loss = portfolio_df['Expected_Loss'].sum()
capital_reserve_ratio = (total_expected_loss / total_credit_exposure) * 100

print("=== FINAL CREDIT RISK MODEL & PORTFOLIO LOSS RESULTS ===")
print(f"Clean Model ROC-AUC Score (No Warnings): {clean_auc:.4f}")
print(f"\nTest Portfolio Size:          {total_test_borrowers:,} borrowers")
print(f"Total Credit Exposure (EAD):   ${total_credit_exposure:,.2f}")
print(f"Total Expected Loss (EL):      ${total_expected_loss:,.2f}")
print(f"Required Statutory Capital:    {capital_reserve_ratio:.2f}% of portfolio")

=== FINAL CREDIT RISK MODEL & PORTFOLIO LOSS RESULTS ===
Clean Model ROC-AUC Score (No Warnings): 0.8107

Test Portfolio Size:          30,000 borrowers
Total Credit Exposure (EAD):   $300,000,000.00
Total Expected Loss (EL):      $9,099,572.64
Required Statutory Capital:    3.03% of portfolio
